# Huntrix — Bengali hallucination detection

This offline inference notebook reproduces the v91 Phase 1 submission and is designed to run unchanged on the Phase 2 held-out set using **Kaggle Tesla T4 ×2**.

The pipeline has two stages:

1. A deterministic evidence router resolves high-confidence source-lineage, structured, lexical, contextual, factual, and legal cases.
2. A pinned Qwen3.6-27B Q4 GGUF judge processes only the unresolved rows and writes one binary label per input row.

The final artifact is `/kaggle/working/submission.csv` with columns exactly `id,label`.

In [ ]:
from __future__ import annotations

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

In [ ]:
import pandas as pd
import torch

## 1. Frozen paths and Phase 2 behavior

**Do not change `TEST_PATH` for Phase 2.** The organizers will replace the dataset mounted at the same required literal path:

```text
/kaggle/input/competitions/bengali-hallucination/test set.csv
```

The number of rows may change from approximately 2,000 to approximately 5,000, but the notebook reads `len(test)` dynamically. No row-count variable, filename search, fallback path, or Phase 1/Phase 2 switch is required.

`RUNTIME`, `MODEL_PATH`, and `SERVER_PATH` are frozen Kaggle asset locations and should only change if the corresponding attached asset itself is deliberately replaced.

In [ ]:
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
TEST_PATH = Path("/kaggle/input/competitions/bengali-hallucination/test set.csv")
RUNTIME = Path(
    "/kaggle/input/datasets/seyamalam/bengali-hallucination-phase2-runtime-v91/"
    "kaggle_phase2_runtime_v91"
)
MODEL_PATH = Path(
    "/kaggle/input/models/snowrab/qwen3-6-27b-q4-k-p/pytorch/default/1/"
    "Qwen3.6-27B-Q4_K_P.gguf"
)
SERVER_PATH = Path(
    "/kaggle/input/datasets/seyamalam/bengali-hallucination-llama-cpp-cuda-b9637-public/"
    "llama-server"
)

## 2. Process helpers

`run` executes a command with fail-fast error handling. `wait_for_server` polls the local llama.cpp health endpoint and stops with a clear error if the model server fails to start within five minutes.

In [ ]:
def run(command: list[str], **kwargs) -> subprocess.CompletedProcess:
    print("+", " ".join(command), flush=True)
    return subprocess.run(command, check=True, text=True, **kwargs)

In [ ]:
def wait_for_server(process: subprocess.Popen, timeout: float = 300.0) -> None:
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"llama-server exited during startup with {process.returncode}")
        try:
            with urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2) as response:
                if response.status == 200:
                    return
        except OSError:
            time.sleep(2)
    raise TimeoutError("llama-server did not become healthy")

## 3. Hardware contract

The frozen runtime was validated on exactly two Tesla T4 GPUs. This guard prevents an accidental run on a single P100 or another untested accelerator from producing an unverifiable result.

In [ ]:
started = time.perf_counter()
if torch.cuda.device_count() != 2 or not all("T4" in torch.cuda.get_device_name(i) for i in range(2)):
    raise RuntimeError(
        f"This submission is validated for T4 x2; observed "
        f"{[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}"
    )

## 4. Input schema and expected dimensions

The test object is a pandas `DataFrame` with shape **`(N, at least 4)`**, where `N` is inferred at runtime. Phase 1 used `N = 2,516`; Phase 2 is expected to use roughly `N = 5,000`.

| Column | Expected dtype | Meaning |
|---|---|---|
| `id` | integer-compatible, unique, ordered | Row identifier copied unchanged to the output |
| `context` | string/object; null permitted | Optional supporting text |
| `prompt_bn` | string/object | Bengali question or instruction |
| `response_bn` | string/object | Bengali response to classify |

Additional columns are tolerated. The notebook fails immediately if any required column is missing.

In [ ]:
test_path = TEST_PATH
test = pd.read_csv(TEST_PATH)
required = {"id", "context", "prompt_bn", "response_bn"}
if not required.issubset(test.columns):
    raise ValueError(f"Bad test file {test_path}: missing {sorted(required - set(test.columns))}")

## 5. Resolve pinned runtime assets

The attached runtime dataset contains the deterministic router, answer banks, Bengali lexical evidence, released benchmark lineage, verified facts, and Bangladesh-law evidence. The model and pinned CUDA `llama-server` are attached separately. This cell validates all three before inference and copies only the executable server into `/kaggle/working`.

In [ ]:
runtime = RUNTIME
scripts = runtime / "scripts"
pipeline = scripts / "phase2_pipeline.py"
model = MODEL_PATH
server = WORK / "llama-server"
for required_asset in (pipeline, model, SERVER_PATH):
    if not required_asset.is_file():
        raise FileNotFoundError(f"Missing pinned runtime asset: {required_asset}")
shutil.copy2(SERVER_PATH, server)
server.chmod(0o755)

## 6. Deterministic evidence router

The router evaluates every row in original order and writes two temporary files:

- `router_submission.csv`: deterministic predictions for resolved rows and a stable base for unresolved rows;
- `router_trace.json`: route decisions plus the uncertainty queue consumed by the judge.

All evidence paths are fixed and offline. The router does not load a saved Phase 1 prediction vector.

In [ ]:
router_output = WORK / "router_submission.csv"
router_trace = WORK / "router_trace.json"
router_command = [
    sys.executable, str(pipeline),
    "--input", str(test_path),
    "--output", str(router_output),
    "--trace", str(router_trace),
    "--bank", str(runtime / "outputs/source_answer_bank_complete_bcs.jsonl"),
    "--bank", str(runtime / "outputs/source_answer_bank_farhan_qdoc.jsonl"),
    "--bank", str(runtime / "outputs/source_answer_bank_bcs_all.jsonl"),
    "--fuzzy-bank", str(runtime / "outputs/source_answer_bank.jsonl"),
    "--lexical-bank", str(runtime / "outputs/lexical_evidence_corpus.jsonl"),
    "--benhallu-gold", str(runtime / "outputs/source_cache/benhallueval_data/qa_gt_1000.csv"),
    "--benhallu-hallucinations", str(runtime / "outputs/source_cache/benhallueval_data/qa_4000.csv"),
    "--verified-facts", str(runtime / "resources/verified_fact_answers.json"),
    "--typed-evidence", str(runtime / "resources/typed_evidence_v1.json"),
    "--idiom-dataset", str(runtime / "outputs/source_cache/bangla_bagdhara_lrec/bagdhara_export_done_2026-03-01_010356.json"),
    "--law-corpus", str(runtime / "outputs/phase2_data/hf_legal_acts_evidence.jsonl"),
    "--law-corpus", str(runtime / "resources/legacy_law_acts.json"),
]
environment = {**os.environ, "PYTHONPATH": str(scripts)}
run(router_command, env=environment)
router_seconds = time.perf_counter() - started

## 7. Selective Qwen judge

llama.cpp loads the 16.332 GiB Qwen3.6-27B Q4_K_P model across both T4 GPUs. Thinking is disabled and generation is constrained to a binary result. Only rows in the router's uncertainty queue are judged.

Context rows use one context-grounding view. No-context rows use up to three fixed prompt views with unanimous-positive voting and early exit after the first negative vote. The `finally` block always stops the server.

In [ ]:
server_log_path = WORK / "llama_server.log"
server_log = server_log_path.open("w", encoding="utf-8")
server_command = [
    str(server),
    "--model", str(model),
    "--host", "127.0.0.1",
    "--port", "8080",
    "--n-gpu-layers", "99",
    "--split-mode", "layer",
    "--tensor-split", "1,1",
    "--ctx-size", "8192",
    "--parallel", "1",
    "--batch-size", "1024",
    "--ubatch-size", "256",
    "--flash-attn", "on",
    "--reasoning-budget", "0",
    "--chat-template-kwargs", '{"enable_thinking":false}',
    "--cache-type-k", "q8_0",
    "--cache-type-v", "q8_0",
    "--no-webui",
]
print("+", " ".join(server_command), flush=True)
server_process = subprocess.Popen(server_command, stdout=server_log, stderr=subprocess.STDOUT, text=True)
try:
    wait_for_server(server_process)
    server_ready_seconds = time.perf_counter() - started
    final_output = WORK / "submission.csv"
    judge_trace = WORK / "judge_trace.json"
    run(
        [
            sys.executable, str(scripts / "gguf_selective_judge.py"),
            "--input", str(test_path),
            "--base-submission", str(router_output),
            "--router-trace", str(router_trace),
            "--output", str(final_output),
            "--trace", str(judge_trace),
            "--workers", "1",
            "--row-batch-size", "4",
        ],
        env=environment,
    )
finally:
    server_process.terminate()
    try:
        server_process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server_process.kill()
    server_log.close()

## 8. Submission contract

Expected output object: a pandas `DataFrame` with shape **`(N, 2)`** and columns exactly:

| Column | Expected dtype | Constraint |
|---|---|---|
| `id` | integer-compatible | Same values and order as the input `id` column |
| `label` | integer | Non-null and restricted to `{0, 1}` |

The CSV is already written to `/kaggle/working/submission.csv`; this cell validates it before the notebook finishes.

In [ ]:
submission = pd.read_csv(WORK / "submission.csv")
if list(submission.columns) != ["id", "label"]:
    raise ValueError(f"Invalid output columns: {submission.columns.tolist()}")
if not submission.id.astype(int).equals(test.id.astype(int)):
    raise ValueError("Output IDs do not match test IDs in order")
if submission.label.isna().any() or not set(submission.label.astype(int)).issubset({0, 1}):
    raise ValueError("Output labels must be non-null binary integers")

## 9. Runtime report

The final cell records row count, GPU names, model size, router/model timing, uncertainty count, label balance, and a linear 5,000-row runtime estimate in `/kaggle/working/phase2_runtime_report.json`. This report is diagnostic and is not part of the submission CSV.

In [ ]:
router_report = json.loads(router_trace.read_text(encoding="utf-8"))
judge_report = json.loads((WORK / "judge_trace.json").read_text(encoding="utf-8"))
elapsed = time.perf_counter() - started
report = {
    "rows": len(test),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(2)],
    "model": str(model),
    "model_gib": round(model.stat().st_size / 1024**3, 3),
    "router_seconds": router_seconds,
    "model_ready_seconds": server_ready_seconds,
    "total_seconds": elapsed,
    "uncertain_rows": router_report["summary"]["uncertain_rows"],
    "judge_changed": judge_report["changed"],
    "label_counts": submission.label.astype(int).value_counts().sort_index().to_dict(),
    "estimated_5000_seconds_at_same_mix": elapsed * 5000 / len(test),
}
(WORK / "phase2_runtime_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(report, ensure_ascii=False, indent=2))